In [2]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score

ROOT = Path('/content/drive/MyDrive/Logika Ui')
if not ROOT.exists():
    from google.colab import drive
    drive.mount('/content/drive')
HASIL = ROOT / 'KTI_KARHUTLA' / 'hasil'
TAU = 0.05

tabel = pd.read_csv(HASIL / 'tabel_gabungan_nasional_semua_bulan.csv',
                    usecols=['NAME_1', 'tahun', 'bulan', 'hujan_rata', 'frac_hujan'])
curah = tabel['hujan_rata'] * 6
kelas = np.where(curah >= 1, 2, np.where(tabel['frac_hujan'] >= TAU, 1, 0))
Y = np.eye(3)[kelas]
tahun = tabel['tahun'].to_numpy()

oof = np.load(HASIL / 'prob_oof_nasional.npz', allow_pickle=True)
klim = {str(k): oof[f'm{i}'] for i, k in enumerate(oof['nama'])}['klimatologi']
fin = np.load(HASIL / 'prob_final_nasional.npz')
PC, PD = fin['PC'], fin['PD']
assert len(PC) == len(tabel) == len(klim), 'jumlah baris tidak sama'


def brier3(P, m):
    P = np.clip(P[m], 1e-6, 1)
    P = P / P.sum(axis=1, keepdims=True)
    return ((P - Y[m]) ** 2).sum(axis=1).mean()


def skill(P, m):
    return 1 - brier3(P, m) / brier3(klim, m)


def kalibrasi_k1(P):
    hasil = P.copy()
    y1 = (kelas == 1).astype(float)
    for th in np.unique(tahun):
        latih, uji = tahun != th, tahun == th
        iso = IsotonicRegression(out_of_bounds='clip').fit(P[latih, 1], y1[latih])
        p1 = iso.predict(P[uji, 1])
        sisa = P[uji, 0] + P[uji, 2]
        hasil[uji, 1] = p1
        hasil[uji, 0] = (1 - p1) * P[uji, 0] / sisa
        hasil[uji, 2] = (1 - p1) * P[uji, 2] / sisa
    return hasil


def reliabilitas(P):
    d = pd.DataFrame({'p': P[:, 1], 'o': (kelas == 1).astype(float)})
    d['desil'] = pd.qcut(d['p'], 10, duplicates='drop')
    g = d.groupby('desil', observed=True)[['p', 'o']].mean().round(3)
    return g.to_numpy().tolist()


def ringkas(P_mentah, P_kal):
    semua = np.ones(len(kelas), bool)
    prov = []
    for p in sorted(tabel['NAME_1'].dropna().unique()):
        m = (tabel['NAME_1'] == p).to_numpy()
        prov.append([p, round(skill(P_kal, m), 4), round(roc_auc_score(kelas[m] == 1, P_kal[m, 1]), 4)])
    return {'sk': round(skill(P_kal, semua), 4),
            'auc': round(roc_auc_score(kelas == 1, P_kal[:, 1]), 4),
            'prov': prov,
            'bulan': [round(skill(P_kal, (tabel['bulan'] == b).to_numpy()), 4) for b in range(1, 13)],
            'kal0': reliabilitas(P_mentah), 'kal1': reliabilitas(P_kal)}


hasil = {'C': ringkas(PC, kalibrasi_k1(PC)), 'D': ringkas(PD, fin['PD_kalibrasi'])}
keluar = HASIL / 'fuwa_cd.js'
keluar.write_text('window.FUWA_CD=' + json.dumps(hasil, separators=(',', ':')) + ';', encoding='utf-8')
print('tersimpan:', keluar)
for m in 'CD':
    print(m, '| skill', hasil[m]['sk'], '| AUC K1', hasil[m]['auc'])

Mounted at /content/drive
tersimpan: /content/drive/MyDrive/Logika Ui/KTI_KARHUTLA/hasil/fuwa_cd.js
C | skill 0.1701 | AUC K1 0.6856
D | skill 0.2103 | AUC K1 0.7048


In [4]:
from pathlib import Path
import geopandas as gpd

ROOT = Path('/content/drive/MyDrive/Logika Ui')
if not ROOT.exists():
    from google.colab import drive
    drive.mount('/content/drive')

g = gpd.read_file(ROOT / 'KTI_KARHUTLA' / 'lainnya' / 'Indonesia_Peat_Lands.geojson')
print(g.shape, g.columns.tolist())
print(g.drop(columns='geometry').head(3).T)

(1524, 8) ['objectid_1', 'objectid', 'layer_revi', 'shape_leng', 'globalid', 'shape_Length', 'shape_Area', 'geometry']
                                                   0  \
objectid_1                                         1   
objectid                                         482   
layer_revi                                    gambut   
shape_leng                              70789.064321   
globalid      {DF27C432-2805-4FCF-8A7E-7E1DFD125C87}   
shape_Length                            72528.694951   
shape_Area                            32863012.50983   

                                                   1  \
objectid_1                                         2   
objectid                                         483   
layer_revi                                    gambut   
shape_leng                               3013.842302   
globalid      {03DD8438-E728-49A1-9DEB-5997D38E1657}   
shape_Length                             2996.905854   
shape_Area                             5